# Unembed Analysis

Analyze the unembedding matrix W_U: spectral structure, token norms,
direction clustering, component projections, and bias effects.

## Why This Matters

Unembedding unembed examines how the model's final representation is projected into vocabulary space to produce predictions. The unembedding matrix is the lens through which all internal computation becomes observable, making its structure fundamental to interpretability.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.unembed_analysis import (
    unembed_spectrum, unembed_token_norms,
    unembed_direction_clustering, unembed_component_projection,
    unembed_bias_analysis,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Unembed Spectrum

SVD of W_U reveals how many effective dimensions the unembedding uses.

In [ ]:
result = unembed_spectrum(model)
print(f"W_U shape: [{result['d_model']}, {result['d_vocab']}]")
print(f"Effective rank: {result['effective_rank']:.2f}")
print(f"Dims for 90% variance: {result['dim_for_90_pct']}")
print(f"Top SV fraction: {result['top_sv_fraction']:.4f}")
print(f"Condition number: {result['condition_number']:.2f}")

## Token Norms

Token columns of W_U vary in norm — outliers may be special tokens.

In [ ]:
result = unembed_token_norms(model, token_ids=[1, 15, 30, 45, 60, 75, 90])
print(f"Global mean norm: {result['global_mean_norm']:.4f}")
print(f"Global std norm: {result['global_std_norm']:.4f}\n")
for t in result['per_token']:
    outlier = ' [OUTLIER]' if t['is_outlier'] else ''
    print(f"  Token {t['token_id']:3d}: norm={t['norm']:.4f}, "
          f"z={t['z_score']:.2f}{outlier}")

## Direction Clustering

Are unembed directions for different tokens well-separated or clustered?

In [ ]:
result = unembed_direction_clustering(model, token_ids=[1, 15, 30, 45, 60])
print(f"Well separated: {result['is_well_separated']}")
print(f"Mean cosine: {result['mean_cosine']:.4f}\n")
for p in result['pairs']:
    print(f"  Token {p['token_a']:2d} vs {p['token_b']:2d}: cos={p['cosine']:.4f}")

## Component Projection

Project each residual stream component through W_U to see which
components drive which token predictions.

In [ ]:
result = unembed_component_projection(model, tokens, layer=0)
print(f"Analyzed {len(result['per_component'])} components\n")
for c in result['per_component']:
    print(f"  {c['component']:20s}: top_token={c['top_token']:3d} "
          f"logit_range={c['logit_range']:.4f} norm={c['projection_norm']:.4f}")

## Bias Analysis

Does the unembedding bias b_U significantly favor certain tokens?

In [ ]:
result = unembed_bias_analysis(model)
print(f"Has significant bias: {result['has_significant_bias']}")
print(f"Bias std: {result['bias_std']:.6f}\n")
print('Top biased tokens:')
for t in result['top_biased_tokens']:
    print(f"  Token {t['token']:3d}: bias={t['bias']:.6f}")